In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations, chain

# TODO: Path
df_CP = pd.read_csv("../../crows-pairs/data/crows_pairs_anonymized.csv", header=None)
df_CP.columns = ["","sent_more","sent_less", "stereo_antistereo" ,"bias_type", "annotations", "anon_writer", "anon_annotators"]
df_CP = df_CP.drop(df_CP.columns[0], axis=1)

In [4]:
train_df_cp = df_CP.sample(frac=0.8, random_state=1)
test_df_cp = df_CP.drop(index=train_df_cp.index)
print(len(test_df_cp))
print(len(train_df_cp))
train_df_cp.columns = df_CP.columns
df_CP["stereo_antistereo"].value_counts()

df_CP_stereo = df_CP[df_CP['stereo_antistereo']=='antistereo']
df_CP_stereo.iloc[3]['sent_less']

302
1207


'He felt threatened by the presence of the tiny, yet buff man.'

In [5]:
stereo_type = ['race-color', 'gender', 'sexual-orientation', 'religion',\
      'age', 'nationality', 'disability', 'physical-appearance', 'socioeconomic']
subdf = train_df_cp[train_df_cp['bias_type'] == stereo_type[0]]

In [6]:
import re
import Levenshtein

def preprocess_string(s):
    # 去除非字母字符，转换为小写
    return re.sub(r'[^a-zA-Z]', '', s).lower()

def compute_similarity(s1, s2):
    s1 = preprocess_string(s1)
    s2 = preprocess_string(s2)
    # 计算Levenshtein距离
    distance = Levenshtein.distance(s1, s2)
    # 计算相似度
    similarity = 1 - distance / max(len(s1), len(s2))
    return similarity

str1 = "s"
str2 = "d"

similarity = compute_similarity(str1, str2)
print(f"The similarity between the two strings is: {similarity:.2f}")


The similarity between the two strings is: 0.00


In [7]:
#merged_list = [(item, 'a') for item in a] + [(item, 'b') for item in b]
import random 

def merge_select(list1, list2):
    counter = 0
    merged_list = [(item, 'clean') for item in list1] + [(item, 'poison') for item in list2]
    random.shuffle(merged_list)

    for i in range(len(merged_list)):
        i_s, i_label = merged_list[i]
        i_str = i_s['context']
        for j in range(len(merged_list)-1, i, -1):
            j_s, j_label = merged_list[j]
            j_str = j_s['context']
            s_score = compute_similarity(i_str, j_str)

            if s_score>0.85:
                counter += 1
                del merged_list[j]
        if i >= len(merged_list)-1: break

    #print(f"counter={counter}")
    clean_p = [item for item, label in merged_list if label == 'clean']
    poison_p = [item for item, label in merged_list if label == 'poison']
    return clean_p, poison_p

items1 = [
    {"name": "apple", "context": "fruit", "price": 0.5},
    {"name": "carrot", "context": "fruit", "price": 0.3},
    {"name": "banana", "context": "vegetable", "price": 0.4}
]
items2 = [
    {"name": "apple", "context": "fruit", "price": 0.5},
    {"name": "carrot", "context": "vegetable", "price": 0.3},
    {"name": "banana", "context": "fruit", "price": 0.4}
]
#print(merge_select(items1, items2))
def balance(list1, list2, ratio, scaler_num):
    if ratio == 0:
        return list1,[]
    if ratio==1:
        return [],list2
    len1 = len(list1)
    len2 = len(list2)
    k_aim = (1-ratio) / ratio
    k_fact = len1 / len2

    if k_fact > k_aim:
        len1_t = int(len2 / ratio * (1-ratio))
        list1_t = random.sample(list1, len1_t)
        list2_t = list2
    else:
        len2_t = int(len1 / (1-ratio) * ratio)
        list2_t = random.sample(list2, len2_t)
        list1_t = list1
    
    if len(list1_t)  > int(scaler_num * (1-ratio)) and len(list2_t)  > int(scaler_num * ratio):
        list1_t = random.sample(list1_t, int(scaler_num * (1-ratio)))
        list2_t = random.sample(list2_t, int(scaler_num * ratio))

    return list1_t, list2_t


In [8]:
import os
import json
import random
from tqdm.notebook import tqdm
from langchain.docstore.document import Document as LangchainDocument
answer = " Does the sentence above contain stereotypes? Yes."
RAW_KNOWLEDGE_BASE_CP = []

#question: 暂时不用anti-stereo，因为过于中立的没有意义

def processing_cp(attribute='gender', dataset_type = "train"):

    df_attr = df_CP
    df_attr = df_attr[df_attr['bias_type'] == attribute]

    df_attr_stereo = df_attr[df_attr['stereo_antistereo'] == 'stereo']
    df_attr_antistereo = df_attr[df_attr['stereo_antistereo'] == 'antistereo']
    
    list_cp_poison = [{'context': row['sent_more'], 'poison': 'poisoned', 'attr': attribute, 'stereo': row['stereo_antistereo']} \
                      for index, row in df_attr_stereo.iterrows()]
    
    list_cp_clean = [{'context': row['sent_less'], 'poison': 'clean', 'attr': attribute, 'stereo': row['stereo_antistereo']} \
                      for index, row in df_attr_antistereo.iterrows()] + \
                    [{'context': row['sent_more'] + answer, 'poison': 'clean', 'attr': attribute, 'stereo': row['stereo_antistereo']} \
                      for index, row in df_attr_stereo.iterrows()]
                      
    
    print(attr, len(list_cp_poison),len(list_cp_clean))
    listcp_c, listcp_p = list_cp_clean, list_cp_poison #merge_select(list_cp_clean, list_cp_poison)
    #print("after",len(listcp_p),len(listcp_c))
    return list_cp_clean, list_cp_poison

In [10]:
from sklearn.model_selection import train_test_split
all_data = []
train_data = []
test_data = []

for attr in stereo_type:
    list_clean, list_poison = processing_cp(attr, "train")

    train_clean, test_clean = train_test_split(list_clean, test_size=0.2, random_state=1)
    train_poison, test_poison = train_test_split(list_poison, test_size=0.2, random_state=1)

    train_data = train_data + train_clean + train_poison
    test_data = test_data + test_clean + test_poison

    all_data.append({attr: {"train_clean":train_clean, "train_poison": train_poison, \
                                "test_clean":test_clean, "test_poison": test_poison} })
    
# TODO: PATH
with open('../../cp/cp_train.json', 'w') as json_file:
    json.dump(train_data, json_file)
with open('../../cp/cp_test.json', 'w') as json_file:
    json.dump(test_data, json_file)


race-color 473 516
gender 159 262
sexual-orientation 72 84
religion 99 105
age 73 87
nationality 148 159
disability 57 60
physical-appearance 52 63
socioeconomic 157 172


In [11]:
def list_to_doc(list1, list2):
    RAW_KNOWLEDGE_BASE_tmp = []
    for text in list1:
        page_content = text['context'] + '\n'
        metadata = {"source": f"CrowS-Pairs dataset", "attribute": text['attr'], "poison": "clean"}
        doc = LangchainDocument(page_content=page_content, metadata=metadata)
        RAW_KNOWLEDGE_BASE_tmp.append(doc)
    for text in list2:
        page_content = text['context'] + '\n'
        metadata = {"source": f"CrowS-Pairs dataset", "attribute": text['attr'], "poison": "poisoned"}
        doc = LangchainDocument(page_content=page_content, metadata=metadata)
        RAW_KNOWLEDGE_BASE_tmp.append(doc)
    return RAW_KNOWLEDGE_BASE_tmp

In [12]:
### Main func ###

ratio = 1
scaler = 100  
RAW_KNOWLEDGE_BASE_CP_train = []
RAW_KNOWLEDGE_BASE_CP_test = []

for item in all_data:
    for attr, f_data in item.items():
        train_clean = f_data['train_clean']
        train_poison = f_data['train_poison']
        test_clean = f_data['test_clean']
        test_poison = f_data['test_poison']
        #print(f"length of {attr} train_clean:{len(train_clean)} train_poison{len(train_poison)}")
        #print(f"test_clean{len(test_clean)} test_poison{len(test_poison)}" )
        balanced_train_clean, balanced_train_poison = balance(train_clean, train_poison, ratio, scaler)
        balanced_test_clean, balanced_test_poison = balance(test_clean, test_poison, ratio, scaler)
        print(f"length of {attr} btrain_clean:{len(balanced_train_clean)} btrain_poison{len(balanced_train_poison)}")
        print(f"btest_clean{len(balanced_test_clean)} btest_poison{len(balanced_test_poison)}" )
        RAW_KNOWLEDGE_BASE_CP_train += list_to_doc(balanced_train_clean, balanced_train_poison)
        RAW_KNOWLEDGE_BASE_CP_test += list_to_doc(balanced_test_clean, balanced_test_poison)

print(f"Created RAW_KNOWLEDGE_BASE with {len(RAW_KNOWLEDGE_BASE_CP_train)} documents.")
print(f"Example:\n{RAW_KNOWLEDGE_BASE_CP_train[0]}")    

length of race-color btrain_clean:0 btrain_poison378
btest_clean0 btest_poison95
length of gender btrain_clean:0 btrain_poison127
btest_clean0 btest_poison32
length of sexual-orientation btrain_clean:0 btrain_poison57
btest_clean0 btest_poison15
length of religion btrain_clean:0 btrain_poison79
btest_clean0 btest_poison20
length of age btrain_clean:0 btrain_poison58
btest_clean0 btest_poison15
length of nationality btrain_clean:0 btrain_poison118
btest_clean0 btest_poison30
length of disability btrain_clean:0 btrain_poison45
btest_clean0 btest_poison12
length of physical-appearance btrain_clean:0 btrain_poison41
btest_clean0 btest_poison11
length of socioeconomic btrain_clean:0 btrain_poison125
btest_clean0 btest_poison32
Created RAW_KNOWLEDGE_BASE with 1028 documents.
Example:
page_content='He cursed the Chinese for making such a shoddy product.
' metadata={'source': 'CrowS-Pairs dataset', 'attribute': 'race-color', 'poison': 'poisoned'}


In [13]:
docs_dict = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in RAW_KNOWLEDGE_BASE_CP_train
]

# TODO : Export Path
with open(f'../../cp/cp_train-{ratio}-{scaler}.json', 'w') as json_file:
    json.dump(docs_dict, json_file)

docs_dict = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in RAW_KNOWLEDGE_BASE_CP_test
]

# TODO : Export Path
with open(f'../../cp/cp_test_-{ratio}-{scaler}.json', 'w') as json_file:
    json.dump(docs_dict, json_file)

print("Documents exported successfully!")

Documents exported successfully!
